This notebook demonstrates a simple Retrieval-Augmented Generation (RAG) pipeline from scratch. The workflow includes:

- PDF text extraction
- Text chunking
- Sentence embeddings
- Cosine similarity retrieval
- Context-aware answer generation using an LLM

This project is intended for educational purposes and serves as a minimal implementation of a PDF Question Answering system.

Install Required Libraries

In [1]:
!pip install pypdf numpy sentence-transformers requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.5 MB/s eta 0:00:00


Import Dependencies

In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load the PDF Document

In [3]:
def load_pdf_text(pdf_path):
  reader = PdfReader(pdf_path)
  full_text = ''
  for page in reader.pages:
    text = page.extract_text()
    if text:
      full_text += text + '\n'
  return full_text

In [16]:
def load_raw_text(story_text):
    if not story_text:
        return ""

    text_str = str(story_text)

    full_text = text_str.strip() + '\n'

    return full_text

Split the Document into Chunks

In [5]:
def custom_text_splitter(text, chunk_size=700, chunk_overlap=100):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += (chunk_size - chunk_overlap)
  return chunks

Load the Embedding Model

In [6]:
embedding_model_name = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(embedding_model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Compute Similarity Scores

In [7]:
def cosine_similarity(vec_a, vec_b):
  dot_product = np.dot(vec_a, vec_b)
  norm_a = np.linalg.norm(vec_a)
  norm_b = np.linalg.norm(vec_b)

  if norm_a == 0 or norm_b == 0:
    return 0
  return dot_product / (norm_a * norm_b)

Retrieve Relevant Context

In [8]:
def retrieve_relevant_chunks(query, chunks, chunk_embeddings, top_k = 2):
  query_embedding = embedding_model.encode(query)

  similarities = []
  for index, chunk_embed in enumerate(chunk_embeddings):
    sim = cosine_similarity(query_embedding, chunk_embed)
    similarities.append((sim, index))

  similarities.sort(key=lambda x: x[0], reverse=True)

  retrieved_context = []
  for sim, index in similarities[:top_k]:
    retrieved_context.append(chunks[index])

  return retrieved_context

Load the Language Model

In [9]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [10]:
model_tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generate the Final Answer

In [18]:
def ask_rag_free(query, chunks, chunk_embeddings):

    context_chunks = retrieve_relevant_chunks(query, chunks, chunk_embeddings, top_k=2)

    context = "\n---\n".join(context_chunks)

    prompt = f"Context: {context}\n\nQuestion: {query}Answer the question based only on the context provided.If you don't know, say \"I don't know.\"Answer:"

    inputs = model_tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1, do_sample=False)

    answer = model_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer.split("Answer:")[-1].strip()

Test the RAG Pipeline

In [19]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [28]:
story_text = gutenberg.raw('carroll-alice.txt')

In [32]:
def question_and_answer(story_text_input, user_query):
  raw_text = load_raw_text(story_text_input)
  chunks = custom_text_splitter(raw_text)
  chunk_embeddings = embedding_model.encode(chunks)

  answer = ask_rag_free(user_query, chunks, chunk_embeddings)
  return answer

In [ ]:
user_query = 'Who does Alice follow down the rabbit hole at the beginning of the story؟'
test = question_and_answer(story_text, user_query)
print(test)

In [27]:
# You can ask the model any question from inside the story